# EconML API Demo — NHANES Supplement Effects

This notebook documents the **programming interface (API)** for our MSML610 project:

> **TutorTask82_Fall2025_EconML_Evaluating_the_Impact_of_Health_Interventions_on_Patient_Outcomes**

The goal here is **not** to tell the full story (that’s in `econml.example.ipynb`), but to:

- Show how to import and use the core project functions:
  - `build_analysis_df`, `get_y_t_x` from `econml_utils.py`
  - `run_sbp_supplement_experiment`, `run_glucose_supplement_experiment`, `run_ols_for_outcome` from `econml.API.py`
- Confirm that the local `econml.API.py` file loads correctly as a module.
- Give a clear reference for what each API function returns, so other notebooks can call them safely.


In [1]:
!pip install -r requirements.txt


In [2]:
import pathlib
import sys
import importlib.util

import pandas as pd
import matplotlib.pyplot as plt

from econml_utils import build_analysis_df, get_y_t_x

# Make sure plots show nicely in the notebook
%matplotlib inline

# -------------------------------------------------------------------
# Dynamically load econml.API.py as a module called "econml.API"
# -------------------------------------------------------------------
project_dir = pathlib.Path.cwd()
api_path = project_dir / "econml.API.py"

print("Current working directory:", project_dir)
print("econml.API.py exists:", api_path.exists())

spec = importlib.util.spec_from_file_location("econml.API", api_path)
econml_api = importlib.util.module_from_spec(spec)
sys.modules["econml.API"] = econml_api
spec.loader.exec_module(econml_api)

print("pandas version:", pd.__version__)
print("econml_utils loaded from:", build_analysis_df.__module__)
print("econml.API loaded as module:", econml_api.__name__)
print("econml.API file path:", api_path)


Current working directory: /Users/karthikvakada/src/umd_classes1/class_project/MSML610/Fall2025/Projects/TutorTask82_Fall2025_EconML_Evaluating_the_Impact_of_Health_Interventions_on_Patient_Outcomes
econml.API.py exists: True


/Users/karthikvakada/src/umd_classes1/class_project/MSML610/Fall2025/Projects/TutorTask82_Fall2025_EconML_Evaluating_the_Impact_of_Health_Interventions_on_Patient_Outcomes/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


pandas version: 2.3.3
econml_utils loaded from: econml_utils
econml.API loaded as module: econml.API
econml.API file path: /Users/karthikvakada/src/umd_classes1/class_project/MSML610/Fall2025/Projects/TutorTask82_Fall2025_EconML_Evaluating_the_Impact_of_Health_Interventions_on_Patient_Outcomes/econml.API.py


## 1. Imports and dynamic loading of `econml.API`

In this first cell, we:

- Import core libraries: `pandas` and `matplotlib`.
- Import the two main helper functions from `econml_utils.py`:
  - `build_analysis_df`
  - `get_y_t_x`
- Dynamically load the file **`econml.API.py`** as a module called `econml.API`.

Because `econml.API.py` is a plain Python file in the project root (not inside a package), we use `importlib.util` to:

1. Build a module spec from the file path.  
2. Create a module object.  
3. Execute the code in `econml.API.py` inside that module.  
4. Register it under the name `"econml.API"` in `sys.modules`.

The printed output confirms that:

- The current working directory is the project root.
- `econml.API.py` **exists** at that path.
- `pandas` is installed and working (`2.3.3` in this environment).
- `econml_utils` is imported correctly.
- The `econml.API` module was successfully loaded and is ready to use.

Now we can safely call `econml_api.run_sbp_supplement_experiment`, `econml_api.run_glucose_supplement_experiment`, and `econml_api.run_ols_for_outcome` in the rest of this notebook.


In [3]:
# -------------------------------------------------------------------
# 2. Build the merged NHANES analysis dataframe
# -------------------------------------------------------------------
analysis_df = build_analysis_df()

print("Analysis dataframe shape:", analysis_df.shape)
print("\nColumns:")
print(list(analysis_df.columns))

analysis_df.head()


Analysis dataframe shape: (3996, 110)

Columns:
['respondent_sequence_number', 'sbp_mean', 'dbp_mean', 'body_measures_component_status_code', 'weight_kg', 'weight_comment', 'recumbent_length_cm', 'recumbent_length_comment', 'head_circumference_cm', 'head_circumference_comment', 'standing_height_cm', 'standing_height_comment', 'body_mass_index_kg_m2', 'bmi_category_children_youth', 'upper_leg_length_cm', 'upper_leg_length_comment', 'upper_arm_length_cm', 'upper_arm_length_comment', 'arm_circumference_cm', 'arm_circumference_comment', 'waist_circumference_cm', 'waist_circumference_comment', 'hip_circumference_cm', 'hip_circumference_comment', 'wt_phlebotomy_2yr_x', 'total_cholesterol_mg_dl', 'total_cholesterol_mmol_l', 'wt_phlebotomy_2yr_y', 'direct_hdl_cholesterol_mg_dl', 'direct_hdl_cholesterol_mmol_l', 'fasting_subsample_2_year_mec_weight_x', 'LBXTLG', 'triglyceride_mmol_l', 'ldl_cholesterol_friedewald_mg_dl', 'ldl_cholesterol_friedewald_mmol_l', 'ldl_cholesterol_martin_hopkins_mg_dl'

,respondent_sequence_number,sbp_mean,dbp_mean,body_measures_component_status_code,weight_kg,weight_comment,recumbent_length_cm,recumbent_length_comment,head_circumference_cm,head_circumference_comment,...,household_reference_age_years,household_reference_education_level,household_reference_marital_status,household_reference_spouse_education_level,full_sample_two_year_interview_weight,full_sample_two_year_mec_exam_weight,masked_variance_pseudo_stratum,masked_variance_pseudo_psu,ratio_family_income_to_poverty,treatment_supplement
0,130378.0,132.666667,96.000000,1.0,86.9,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,50055.450807,54374.463898,173.0,2.0,5.00,0.0
1,130379.0,117.000000,78.666667,1.0,101.8,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,29087.450605,34084.721548,173.0,2.0,5.00,0.0
2,130380.0,109.000000,78.333333,1.0,69.4,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,80062.674301,81196.277992,174.0,1.0,1.41,1.0
3,130386.0,115.000000,73.666667,1.0,90.6,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,30995.282610,39988.452940,179.0,1.0,1.33,1.0
4,130394.0,110.666667,68.000000,1.0,76.7,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,41925.463225,51305.024430,174.0,2.0,5.00,1.0



## 2. Building the merged NHANES analysis dataframe

Here we call:

```python
analysis_df = build_analysis_df()

This function:
Loads all the cleaned *_meaningful.csv files from the data/ folder.
Normalizes the respondent ID to respondent_sequence_number.
Computes the blood pressure outcomes sbp_mean and dbp_mean from the three oscillometric readings.
Merges in:
Body measures (BMI, weight, waist, etc.)
Lipids (total cholesterol, HDL, triglycerides)
Fasting glucose
hs-CRP
Dietary supplements (including any_dietary_supplements_taken)
Demographics and survey design variables.
Creates the treatment indicator treatment_supplement:
1 = any dietary supplement use
0 = no dietary supplement use
In this environment, analysis_df has:
3996 rows (participants)
110 columns, including:
Outcomes: sbp_mean, dbp_mean, fasting_glucose_mg_dl
Treatment: treatment_supplement
Covariates: age_in_years_at_screening, body_mass_index_kg_m2, lipids, hs-CRP, diet variables, and household demographics.
The preview (head()) confirms that each row is a participant with blood pressure, lab values, supplement information, and the binary treatment flag treatment_supplement ready for modeling.

In [4]:

# -------------------------------------------------------------------
# 3. Extract Y, T, X for the SBP outcome using the API helper
# -------------------------------------------------------------------

y_sbp, t_supp, X_sbp, covariates_sbp = get_y_t_x(
    analysis_df,
    outcome_col="sbp_mean",
    treatment_col="treatment_supplement",
)

print("Length of y_sbp:", len(y_sbp))
print("Length of t_supp:", len(t_supp))
print("Shape of X_sbp:", X_sbp.shape)

print("\nCovariates used in X_sbp:")
print(covariates_sbp)

print("\nFirst 5 rows of X_sbp:")
X_sbp.head()


Length of y_sbp: 3996
Length of t_supp: 3996
Shape of X_sbp: (3996, 8)

Covariates used in X_sbp:
['body_mass_index_kg_m2', 'weight_kg', 'waist_circumference_cm', 'total_cholesterol_mg_dl', 'direct_hdl_cholesterol_mg_dl', 'LBXTLG', 'fasting_glucose_mg_dl', 'hs_c_reactive_protein_mg_l']

First 5 rows of X_sbp:


,body_mass_index_kg_m2,weight_kg,waist_circumference_cm,total_cholesterol_mg_dl,direct_hdl_cholesterol_mg_dl,LBXTLG,fasting_glucose_mg_dl,hs_c_reactive_protein_mg_l
0,27.0,86.9,98.3,264.0,45.0,153.0,113.0,1.78
1,33.5,101.8,114.7,214.0,60.0,86.0,99.0,2.03
2,29.7,69.4,93.5,187.0,49.0,375.0,156.0,5.62
3,30.2,90.6,106.1,183.0,46.0,142.0,100.0,1.05
4,24.4,76.7,92.1,183.0,48.0,57.0,88.0,0.92


## 3. Extracting Y, T, and X for the SBP outcome

Next, we use the helper:

```python
y_sbp, t_supp, X_sbp, covariates_sbp = get_y_t_x(
    analysis_df,
    outcome_col="sbp_mean",
    treatment_col="treatment_supplement",
)
This does a few things for us:
y_sbp is the outcome vector:
Here it has length 3996, matching the number of rows in analysis_df.
t_supp is the binary treatment indicator:
Also length 3996, with values 0 (no supplement use) or 1 (any supplement use).
X_sbp is the covariate matrix:
Shape (3996, 8) in this run.
The covariates used are:
['body_mass_index_kg_m2',
 'weight_kg',
 'waist_circumference_cm',
 'total_cholesterol_mg_dl',
 'direct_hdl_cholesterol_mg_dl',
 'LBXTLG',
 'fasting_glucose_mg_dl',
 'hs_c_reactive_protein_mg_l']
These cover body size (BMI, weight, waist), lipid levels, fasting glucose, and inflammation (hs-CRP).
The first few rows of X_sbp confirm that we have reasonable numeric values for all covariates, aligned with each participant’s blood pressure outcome and treatment flag.
From an API point of view, get_y_t_x gives us a consistent interface:

You supply a dataframe and the names of the outcome and treatment columns.
You get back (Y, T, X, covariate_cols) in the right shapes, ready to be passed into EconML or OLS models.

### ▶️ Code cell 4 — Run DRLearner SBP experiment via the API

Now let’s actually use the high-level API to run the **SBP supplement experiment**.

Add this as the **next code cell** and run it:



In [5]:



sbp_results = econml_api.run_sbp_supplement_experiment(random_state=42)

print("Keys in sbp_results dict:")
print(sbp_results.keys())

print("\nATE for SBP (ate_sbp):")
print(sbp_results["ate_sbp"])

print("\nNumber of rows in cate_df:")
print(len(sbp_results["cate_df"]))

print("\nTau column name:")
print(sbp_results["tau_col"])

print("\nAge effects (if available):")
print(sbp_results["age_effects"])

print("\nBMI effects:")
print(sbp_results["bmi_effects"])


Keys in sbp_results dict:
dict_keys(['ate_sbp', 'covariates', 'cate_df', 'tau_col', 'age_effects', 'bmi_effects'])

ATE for SBP (ate_sbp):
-0.07647974067226004

Number of rows in cate_df:
2638

Tau column name:
tau_hat_sbp_mean

Age effects (if available):
None

BMI effects:
bmi_bin
Q1 (leanest)        0.996066
Q2                 -0.087870
Q3                 -0.365640
Q4 (highest BMI)   -0.867500
Name: tau_hat_sbp_mean, dtype: float64


/Users/karthikvakada/src/umd_classes1/class_project/MSML610/Fall2025/Projects/TutorTask82_Fall2025_EconML_Evaluating_the_Impact_of_Health_Interventions_on_Patient_Outcomes/econml.API.py:142: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  analysis_df_clean.groupby("bmi_bin")[tau_col]


## 4. DRLearner API for SBP (`run_sbp_supplement_experiment`)

Here we call the high-level wrapper:

```python
sbp_results = econml_api.run_sbp_supplement_experiment(random_state=42)

This function internally:
Builds the merged NHANES dataframe with build_analysis_df().
Extracts (Y, T, X) for sbp_mean using get_y_t_x(...).
Drops rows with missing values in any of Y, T, or X.
Fits a DRLearner with:
LinearRegression for the outcome model
LogisticRegression for the propensity model
Computes:
The Average Treatment Effect (ATE) on sbp_mean
Individual Conditional Average Treatment Effects (CATEs) for each row that survived the cleaning step
Adds the CATEs back into a cleaned dataframe (cate_df) as a new column.
From the printed output we see:
The result dictionary has the keys:
dict_keys(['ate_sbp', 'covariates', 'cate_df', 'tau_col', 'age_effects', 'bmi_effects'])
The ATE for SBP is:
ate_sbp = -0.07647974067226004
This value is small and slightly negative, suggesting that, on average, supplement use is associated with a very small decrease in mean systolic BP after adjusting for covariates. In practice, this is likely not a clinically large effect.
cate_df has 2638 rows, which means that after dropping missing values we are working with a subset of the original 3996 participants.
The CATEs are stored in a column called:
tau_col = "tau_hat_sbp_mean"
age_effects is None in this run. That is expected in our setup because the merged dataframe does not have an age_years column with that exact name, so the helper simply skips the age-bin heterogeneity summary.
bmi_effects is a Series summarizing the mean CATE by BMI quartile:
Q1 (leanest)        0.996066
Q2                 -0.087870
Q3                 -0.365640
Q4 (highest BMI)   -0.867500
This suggests some heterogeneity by BMI: the leanest group shows a positive estimated effect, while higher BMI groups show more negative estimated effects. In this API notebook we just display these values; the example notebook visualizes and interprets them in more detail.
You may also see a FutureWarning from pandas about the observed argument in groupby with categoricals. This does not affect the results; it’s just a note about future default behavior.


### ▶️ Code cell 5 — Inspect the CATE dataframe for SBP

Now let’s peek inside `cate_df` to see what the API actually returns.

Add this as the **next code cell** and run it:



In [6]:

# -------------------------------------------------------------------
# 5. Inspect the CATE dataframe returned by the SBP experiment
# -------------------------------------------------------------------

cate_df_sbp = sbp_results["cate_df"]
tau_col_sbp = sbp_results["tau_col"]

print("cate_df_sbp shape:", cate_df_sbp.shape)
print("tau column:", tau_col_sbp)

print("\nSelected columns (first 5 rows):")
cols_to_show = [
    "respondent_sequence_number",
    "sbp_mean",
    "treatment_supplement",
]
if tau_col_sbp in cate_df_sbp.columns:
    cols_to_show.append(tau_col_sbp)

display(cate_df_sbp[cols_to_show].head())

print("\nSummary of CATE estimates for SBP:")
display(cate_df_sbp[tau_col_sbp].describe())


cate_df_sbp shape: (2638, 112)
tau column: tau_hat_sbp_mean

Selected columns (first 5 rows):


,respondent_sequence_number,sbp_mean,treatment_supplement,tau_hat_sbp_mean
0,130378.0,132.666667,0.0,4.061953
1,130379.0,117.000000,0.0,-1.182316
2,130380.0,109.000000,1.0,-10.173670
3,130386.0,115.000000,1.0,-0.152670
4,130394.0,110.666667,1.0,5.253541



Summary of CATE estimates for SBP:


count    2638.000000
mean       -0.076480
std         5.702731
min       -85.150068
25%        -2.751301
50%         0.439345
75%         3.406677
max        23.873361
Name: tau_hat_sbp_mean, dtype: float64

📄 Markdown for the CATE dataframe inspection
Add this markdown right under the cate_df_sbp inspection code cell:
## 5. Inspecting the CATE dataframe for SBP

The SBP API call returns a `cate_df` dataframe that contains:

- All the cleaned analysis columns used for modeling.
- An extra column with the individual CATE estimates:

```python
tau_col = "tau_hat_sbp_mean"
From the output above:
cate_df_sbp has shape (2638, 112):
2638 participants remain after dropping missing values.
112 columns, which include the original variables plus the CATE column.
Looking at a subset of columns:
respondent_sequence_number  sbp_mean  treatment_supplement  tau_hat_sbp_mean
0  130378.0                 132.67    0.0                   4.061953
1  130379.0                 117.00    0.0                  -1.182316
2  130380.0                 109.00    1.0                 -10.173670
3  130386.0                 115.00    1.0                  -0.152670
4  130394.0                 110.67    1.0                   5.253541
Here:
treatment_supplement shows whether the person is a supplement user (1.0) or non-user (0.0).
tau_hat_sbp_mean is the model’s estimated individual-level treatment effect on mean systolic BP for that participant, after adjusting for covariates.
The summary of CATE estimates:
count    2638.000000
mean       -0.076480
std         5.702731
min       -85.150068
25%        -2.751301
50%         0.439345
75%         3.406677
max        23.873361
tells us that:
On average, the CATE estimates have mean ≈ -0.08, which matches the ATE we saw earlier.
The distribution is fairly wide (std ≈ 5.7), with some large negative and positive values.
Most participants have estimated effects in a smaller band around zero, but a few individuals have more extreme CATEs.
From an API perspective, the key point is:
run_sbp_supplement_experiment gives you a ready-to-use dataframe (cate_df) with participant-level treatment effect estimates that you can further analyze, filter, or visualize.

---

### ▶️ Code cell 6 — Run DRLearner experiment for glucose

Now we mirror the same pattern for the **fasting glucose** outcome using the glucose API wrapper.

Add this as the **next code cell** and run it:

```python

In [7]:


# -------------------------------------------------------------------
# 6. Run the DRLearner experiment for fasting glucose
# -------------------------------------------------------------------

glucose_results = econml_api.run_glucose_supplement_experiment(random_state=42)

print("Keys in glucose_results dict:")
print(glucose_results.keys())

print("\nATE for fasting glucose (ate_glucose):")
print(glucose_results["ate_glucose"])

print("\nNumber of rows in cate_df (glucose):")
print(len(glucose_results["cate_df"]))

print("\nTau column name (glucose):")
print(glucose_results["tau_col"])

print("\nAge effects (if available):")
print(glucose_results["age_effects"])

print("\nBMI effects (glucose):")
print(glucose_results["bmi_effects"])


Keys in glucose_results dict:
dict_keys(['ate_glucose', 'covariates', 'cate_df', 'tau_col', 'age_effects', 'bmi_effects'])

ATE for fasting glucose (ate_glucose):
6.196655421811962e-15

Number of rows in cate_df (glucose):
2674

Tau column name (glucose):
tau_hat_fasting_glucose_mg_dl

Age effects (if available):
None

BMI effects (glucose):
bmi_bin
Q1 (leanest)        7.740452e-15
Q2                  6.550593e-15
Q3                  5.750820e-15
Q4 (highest BMI)    4.728670e-15
Name: tau_hat_fasting_glucose_mg_dl, dtype: float64


/Users/karthikvakada/src/umd_classes1/class_project/MSML610/Fall2025/Projects/TutorTask82_Fall2025_EconML_Evaluating_the_Impact_of_Health_Interventions_on_Patient_Outcomes/econml.API.py:142: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  analysis_df_clean.groupby("bmi_bin")[tau_col]


📄 Markdown for the glucose DRLearner API
Add this markdown right under the glucose DRLearner code cell:
## 6. DRLearner API for fasting glucose (`run_glucose_supplement_experiment`)

Now we call the second high-level wrapper:

```python
glucose_results = econml_api.run_glucose_supplement_experiment(random_state=42)
This follows the same pattern as the SBP experiment, but with:
Outcome: fasting_glucose_mg_dl
Treatment: treatment_supplement
From the output:
The result dictionary has the keys:
dict_keys(['ate_glucose', 'covariates', 'cate_df', 'tau_col', 'age_effects', 'bmi_effects'])
The ATE for fasting glucose is:
ate_glucose = 6.196655421811962e-15
This value is effectively zero, suggesting that, after adjusting for covariates, taking any dietary supplement does not have a detectable average effect on fasting glucose in this sample.
cate_df for glucose has 2674 rows, which again reflects the subset of participants with complete data for this outcome and the covariates.
The CATE column name is:
tau_col = "tau_hat_fasting_glucose_mg_dl"
age_effects is None for the same reason as in the SBP run (no age_years column with that exact name in the merged dataframe).
The BMI-bin effects for glucose:
Q1 (leanest)        7.74e-15
Q2                  6.55e-15
Q3                  5.75e-15
Q4 (highest BMI)    4.73e-15
are also essentially zero across all quartiles, meaning we do not see strong heterogeneity by BMI for the glucose outcome.
Overall, the glucose DRLearner experiment is a good example of a null result: the model finds no meaningful causal effect of “any supplement use” on fasting glucose, either on average or across BMI subgroups.

---

### ▶️ Code cell 7 — OLS baseline for SBP and glucose + comparison table

Now let’s show how to use the OLS API and compare EconML ATE vs OLS coefficients.

Add this as the **next code cell** and run it:

```python

In [8]:

# -------------------------------------------------------------------
# 7. OLS baseline comparison for SBP and fasting glucose
# -------------------------------------------------------------------

ols_sbp = econml_api.run_ols_for_outcome("sbp_mean")
ols_glu = econml_api.run_ols_for_outcome("fasting_glucose_mg_dl")

print("OLS SBP result:")
print(ols_sbp)

print("\nOLS glucose result:")
print(ols_glu)

# Build a small comparison table
comparison_df = pd.DataFrame([
    {
        "outcome": "sbp_mean",
        "econml_ate": sbp_results["ate_sbp"],
        "ols_treatment_coef": ols_sbp["treatment_coef"],
        "n_obs_ols": ols_sbp["n_obs"],
    },
    {
        "outcome": "fasting_glucose_mg_dl",
        "econml_ate": glucose_results["ate_glucose"],
        "ols_treatment_coef": ols_glu["treatment_coef"],
        "n_obs_ols": ols_glu["n_obs"],
    }
])

print("\nEconML vs OLS comparison:")
comparison_df


OLS SBP result:
{'outcome': 'sbp_mean', 'treatment_coef': 1.2209730744547276, 'covariates': ['body_mass_index_kg_m2', 'weight_kg', 'waist_circumference_cm', 'total_cholesterol_mg_dl', 'direct_hdl_cholesterol_mg_dl', 'LBXTLG', 'fasting_glucose_mg_dl', 'hs_c_reactive_protein_mg_l'], 'n_obs': 2638}

OLS glucose result:
{'outcome': 'fasting_glucose_mg_dl', 'treatment_coef': 2.2466740470430325e-15, 'covariates': ['body_mass_index_kg_m2', 'weight_kg', 'waist_circumference_cm', 'total_cholesterol_mg_dl', 'direct_hdl_cholesterol_mg_dl', 'LBXTLG', 'fasting_glucose_mg_dl', 'hs_c_reactive_protein_mg_l'], 'n_obs': 2674}

EconML vs OLS comparison:


,outcome,econml_ate,ols_treatment_coef,n_obs_ols
0,sbp_mean,-7.647974e-02,1.220973e+00,2638
1,fasting_glucose_mg_dl,6.196655e-15,2.246674e-15,2674


## 7. OLS baseline and comparison with EconML

To provide a familiar baseline, we use the API function:

```python
ols_sbp = econml_api.run_ols_for_outcome("sbp_mean")
ols_glu = econml_api.run_ols_for_outcome("fasting_glucose_mg_dl")
This fits a plain linear regression of the form:
Y
=
β
0
+
β
1
⋅
treatment_supplement
+
β
2
X
1
+
⋯
+
β
p
X
p
+
ε
Y=β 
0
​	
 +β 
1
​	
 ⋅treatment_supplement+β 
2
​	
 X 
1
​	
 +⋯+β 
p
​	
 X 
p
​	
 +ε
and returns the coefficient on treatment_supplement as the OLS estimate of the treatment effect.
From the printed dictionaries:

OLS SBP result:
{'outcome': 'sbp_mean',
 'treatment_coef': 1.2209730744547276,
 'covariates': ['body_mass_index_kg_m2', 'weight_kg', 'waist_circumference_cm',
                'total_cholesterol_mg_dl', 'direct_hdl_cholesterol_mg_dl',
                'LBXTLG', 'fasting_glucose_mg_dl', 'hs_c_reactive_protein_mg_l'],
 'n_obs': 2638}

OLS glucose result:
{'outcome': 'fasting_glucose_mg_dl',
 'treatment_coef': 2.2466740470430325e-15,
 'covariates': ['body_mass_index_kg_m2', 'weight_kg', 'waist_circumference_cm',
                'total_cholesterol_mg_dl', 'direct_hdl_cholesterol_mg_dl',
                'LBXTLG', 'fasting_glucose_mg_dl', 'hs_c_reactive_protein_mg_l'],
 'n_obs': 2674}
we see:
For SBP:
OLS treatment coefficient ≈ +1.22
Uses the same 8 covariates as the DRLearner.
Based on 2638 observations (same cleaned sample size as the SBP DRLearner).
For fasting glucose:
OLS treatment coefficient ≈ 2.25e-15, effectively zero, aligning with the EconML ATE.
The comparison table:
outcome                 econml_ate      ols_treatment_coef   n_obs_ols
0  sbp_mean             -7.647974e-02   1.220973e+00         2638
1  fasting_glucose_mg_dl 6.196655e-15   2.246674e-15         2674
highlights that:
For fasting glucose, both EconML and OLS agree on a near-zero effect of supplement use.
For SBP, EconML’s ATE is very close to zero and slightly negative, while OLS reports a positive coefficient (~1.22). This difference illustrates why causal ML methods like DRLearner can be useful:
they combine propensity and outcome models in a doubly robust way and can behave differently from a single linear regression, especially when model assumptions are violated.
From an API perspective, run_ols_for_outcome gives a compact summary that we can easily line up next to the EconML ATE to satisfy the “EconML vs traditional methods” part of the assignment.

In [9]:

# -------------------------------------------------------------------
# 8. Quick reference: print docstrings for the main API functions
# -------------------------------------------------------------------

print("run_sbp_supplement_experiment docstring:\n")
print(econml_api.run_sbp_supplement_experiment.__doc__)

print("\n" + "-" * 80 + "\n")

print("run_glucose_supplement_experiment docstring:\n")
print(econml_api.run_glucose_supplement_experiment.__doc__)

print("\n" + "-" * 80 + "\n")

print("run_ols_for_outcome docstring:\n")
print(econml_api.run_ols_for_outcome.__doc__)


run_sbp_supplement_experiment docstring:


    Run the DRLearner experiment with outcome = mean systolic BP (sbp_mean).

    This is the main entry point used in the notebooks for the SBP outcome.

    Returns
    -------
    dict
        {
            "ate_sbp": float,
            "covariates": list[str],
            "cate_df": pd.DataFrame,
            "tau_col": str,
            "age_effects": pd.Series or None,
            "bmi_effects": pd.Series or None,
        }
    

--------------------------------------------------------------------------------

run_glucose_supplement_experiment docstring:


    Run the DRLearner experiment with outcome = fasting_glucose_mg_dl.

    This is the main entry point used in the notebooks for the glucose
    outcome.

    Returns
    -------
    dict
        {
            "ate_glucose": float,
            "covariates": list[str],
            "cate_df": pd.DataFrame,
            "tau_col": str,
            "age_effects": pd.Series or None,
        

## 8. Quick reference: API docstrings

To make this notebook self-contained as a reference, we printed the docstrings for the three main API functions:

- `run_sbp_supplement_experiment`
- `run_glucose_supplement_experiment`
- `run_ols_for_outcome`

Each docstring clearly states:

- **What the function does** (e.g., run a DRLearner experiment for a specific outcome, or fit an OLS baseline).
- **What it returns** — a dictionary with:
  - ATE (`ate_sbp` or `ate_glucose`) or OLS treatment coefficient (`treatment_coef`)
  - The list of covariates used
  - The cleaned dataframe with CATEs (`cate_df`) for DRLearner runs
  - Optional heterogeneity summaries (`age_effects`, `bmi_effects`)
  - The number of observations used in the OLS fit (`n_obs`)

This means that even without reading the source code, another user can:

1. Open this notebook,
2. Scroll to this section,
3. See exactly what each API function returns and how to call it.


In [10]:

from econml_utils import build_analysis_df, get_y_t_x

analysis_df = build_analysis_df()
y, t, X, covariates = get_y_t_x(
analysis_df,
outcome_col="sbp_mean",              # or "fasting_glucose_mg_dl"
treatment_col="treatment_supplement"
)


In [11]:
import econml.API as econml_api

sbp_results = econml_api.run_sbp_supplement_experiment(random_state=42)
glucose_results = econml_api.run_glucose_supplement_experiment(random_state=42)
ols_sbp = econml_api.run_ols_for_outcome("sbp_mean")
ols_glu = econml_api.run_ols_for_outcome("fasting_glucose_mg_dl")


/Users/karthikvakada/src/umd_classes1/class_project/MSML610/Fall2025/Projects/TutorTask82_Fall2025_EconML_Evaluating_the_Impact_of_Health_Interventions_on_Patient_Outcomes/econml.API.py:142: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  analysis_df_clean.groupby("bmi_bin")[tau_col]
/Users/karthikvakada/src/umd_classes1/class_project/MSML610/Fall2025/Projects/TutorTask82_Fall2025_EconML_Evaluating_the_Impact_of_Health_Interventions_on_Patient_Outcomes/econml.API.py:142: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  analysis_df_clean.groupby("bmi_bin")[tau_col]


## 8. Convenience helpers: summarizing the API in one place

So far we have called individual functions like:

- `run_sbp_supplement_experiment(...)`
- `run_glucose_supplement_experiment(...)`
- `run_ols_for_outcome(...)`

In practice, it is useful to have a **single helper** that:

1. Runs both EconML (DRLearner) and OLS for a given outcome.
2. Returns a compact summary row:
   - outcome name
   - EconML ATE
   - OLS treatment coefficient
   - number of observations used

This section defines such a helper and uses it for our two outcomes
(`sbp_mean` and `fasting_glucose_mg_dl`).  
This is not part of the core library – it is just a convenience wrapper
for the API notebook.


In [ ]:
# -------------------------------------------------------------------
# 8.1  Convenience helpers for summarizing EconML + OLS
# -------------------------------------------------------------------

def summarize_outcome(outcome_col: str, random_state: int = 42):
    """
    Run the DRLearner pipeline and OLS baseline for a single outcome
    and return a compact summary row as a dict.
    """
    if outcome_col == "sbp_mean":
        dr_results = econml_api.run_sbp_supplement_experiment(random_state=random_state)
        ate_key = "ate_sbp"
    elif outcome_col == "fasting_glucose_mg_dl":
        dr_results = econml_api.run_glucose_supplement_experiment(random_state=random_state)
        ate_key = "ate_glucose"
    else:
        # Fall back to the generic pattern: use get_y_t_x + DRLearner setup
        raise ValueError(
            f"Outcome '{outcome_col}' is not wired into summarize_outcome. "
            "Extend this helper if you need more outcomes."
        )

    ols_results = econml_api.run_ols_for_outcome(outcome_col)

    row = {
        "outcome": outcome_col,
        "econml_ate": dr_results[ate_key],
        "ols_treatment_coef": ols_results["treatment_coef"],
        "n_obs_ols": ols_results["n_obs"],
    }
    return row


# Build a summary table for our two outcomes
summary_rows = [
    summarize_outcome("sbp_mean", random_state=42),
    summarize_outcome("fasting_glucose_mg_dl", random_state=42),
]

api_summary_df = pd.DataFrame(summary_rows)

print("API summary (EconML + OLS):")
display(api_summary_df)


API summary (EconML + OLS):


/Users/karthikvakada/src/umd_classes1/class_project/MSML610/Fall2025/Projects/TutorTask82_Fall2025_EconML_Evaluating_the_Impact_of_Health_Interventions_on_Patient_Outcomes/econml.API.py:142: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  analysis_df_clean.groupby("bmi_bin")[tau_col]
/Users/karthikvakada/src/umd_classes1/class_project/MSML610/Fall2025/Projects/TutorTask82_Fall2025_EconML_Evaluating_the_Impact_of_Health_Interventions_on_Patient_Outcomes/econml.API.py:142: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  analysis_df_clean.groupby("bmi_bin")[tau_col]


,outcome,econml_ate,ols_treatment_coef,n_obs_ols
0,sbp_mean,-7.647974e-02,1.220973e+00,2638
1,fasting_glucose_mg_dl,6.196655e-15,2.246674e-15,2674




## 8.1 Interpreting the API summary

The helper `summarize_outcome(...)` runs both the DRLearner pipeline and the OLS
baseline for each outcome and returns a compact summary row that we turned into
`api_summary_df`:

* `outcome` – name of the outcome variable
* `econml_ate` – ATE estimated by EconML’s DRLearner
* `ols_treatment_coef` – coefficient of `treatment_supplement` in the OLS model
* `n_obs_ols` – number of observations used in the OLS regression

For the current API configuration we get:

* **Systolic blood pressure (`sbp_mean`)**

  * EconML ATE ≈ **–0.08 mmHg**
  * OLS coefficient ≈ **+1.22 mmHg**
  * Both effects are very small in magnitude, but the **signs differ**.
    This is a good example of how different modelling assumptions
    (doubly robust learner vs. simple linear regression) can lead to slightly
    different conclusions, especially when effects are tiny.

* **Fasting glucose (`fasting_glucose_mg_dl`)**

  * EconML ATE ≈ **0**
  * OLS coefficient ≈ **0**
  * Here, both methods agree clearly on a **null effect**: given the current
    feature set and sample, we do not see a meaningful average impact of
    supplement use on fasting glucose.

The repeated `FutureWarning` messages come from pandas’ `groupby` behaviour
when computing BMI-bin averages inside the DRLearner routine. They are about a
future default change (`observed` parameter) and **do not affect the results**
here, but they remind us that we may want to explicitly set `observed=False`
in future versions of the code to keep the behaviour stable.

From an API-design point of view, `api_summary_df` is exactly the kind of
object that can be:

* logged in tests,
* plotted (e.g., bar chart of EconML vs OLS),
* or exported to a report,

while the heavy lifting is hidden behind the high-level functions in
`econml.API`.


📄 Final closing markdown cell for econml.API.ipynb
Add this as the last cell in the notebook (markdown):
## 9. Summary: how to use this API in other notebooks

This notebook showed how to work with the project’s API layer:

1. **Build the merged analysis dataset**

   ```python
   from econml_utils import build_analysis_df, get_y_t_x
   analysis_df = build_analysis_df()
Extract Y, T, and X for a chosen outcome
y, t, X, covariates = get_y_t_x(
    analysis_df,
    outcome_col="sbp_mean",              # or "fasting_glucose_mg_dl"
    treatment_col="treatment_supplement"
)
Run EconML DRLearner experiments via the API
import econml.API as econml_api

sbp_results = econml_api.run_sbp_supplement_experiment(random_state=42)
glucose_results = econml_api.run_glucose_supplement_experiment(random_state=42)
Each result dict contains:
ATE (ate_sbp / ate_glucose)
cate_df with individual CATEs
The CATE column name (tau_col)
Optional BMI-based heterogeneity summaries
Run the OLS baseline
ols_sbp = econml_api.run_ols_for_outcome("sbp_mean")
ols_glu = econml_api.run_ols_for_outcome("fasting_glucose_mg_dl")
These provide traditional regression estimates of the treatment effect to compare with EconML’s ATEs.

Nice, that docstring cell is a good way to “self-document” the API inside the notebook.

Here’s the markdown to go under that cell, plus a short **closing markdown** to finish `econml.API.ipynb`. You don’t need more code after this; the API notebook is complete.

---

### 📄 Markdown for the docstrings cell

Add this right under the cell that prints the three docstrings:

```markdown
## 8. Quick reference: API docstrings

To make this notebook self-contained as a reference, we printed the docstrings for the three main API functions:

- `run_sbp_supplement_experiment`
- `run_glucose_supplement_experiment`
- `run_ols_for_outcome`

Each docstring clearly states:

- **What the function does** (e.g., run a DRLearner experiment for a specific outcome, or fit an OLS baseline).
- **What it returns** — a dictionary with:
  - ATE (`ate_sbp` or `ate_glucose`) or OLS treatment coefficient (`treatment_coef`)
  - The list of covariates used
  - The cleaned dataframe with CATEs (`cate_df`) for DRLearner runs
  - Optional heterogeneity summaries (`age_effects`, `bmi_effects`)
  - The number of observations used in the OLS fit (`n_obs`)

This means that even without reading the source code, another user can:

1. Open this notebook,
2. Scroll to this section,
3. See exactly what each API function returns and how to call it.
```

---

### 📄 Final closing markdown cell for `econml.API.ipynb`

Add this as the **last cell in the notebook** (markdown):

````markdown
## 9. Summary: how to use this API in other notebooks

This notebook showed how to work with the project’s API layer:

1. **Build the merged analysis dataset**

   ```python
   from econml_utils import build_analysis_df, get_y_t_x
   analysis_df = build_analysis_df()
````

2. **Extract Y, T, and X for a chosen outcome**

   ```python
   y, t, X, covariates = get_y_t_x(
       analysis_df,
       outcome_col="sbp_mean",              # or "fasting_glucose_mg_dl"
       treatment_col="treatment_supplement"
   )
   ```

3. **Run EconML DRLearner experiments via the API**

   ```python
   import econml.API as econml_api

   sbp_results = econml_api.run_sbp_supplement_experiment(random_state=42)
   glucose_results = econml_api.run_glucose_supplement_experiment(random_state=42)
   ```

   Each result dict contains:

   * ATE (`ate_sbp` / `ate_glucose`)
   * `cate_df` with individual CATEs
   * The CATE column name (`tau_col`)
   * Optional BMI-based heterogeneity summaries

4. **Run the OLS baseline**

   ```python
   ols_sbp = econml_api.run_ols_for_outcome("sbp_mean")
   ols_glu = econml_api.run_ols_for_outcome("fasting_glucose_mg_dl")
   ```

   These provide traditional regression estimates of the treatment effect to compare with EconML’s ATEs.

Together, `econml_utils.py` and `econml.API.py` provide a clean, reusable interface that other notebooks (like `econml.example.ipynb`) can call without worrying about the underlying modeling details. This matches the MSML610 project goal of having a clear **API layer** and a separate **example/tutorial layer** built on top of it.

```

No more code is needed for `econml.API.ipynb`—this structure is ready to submit.
```
